# Image → 3D on Colab (Stable Fast 3D, free T4)

Turns a **single image** into a clean, UV-unwrapped, **textured** 3D mesh using
[Stable Fast 3D (SF3D)](https://github.com/Stability-AI/stable-fast-3d) on a free Colab T4 —
**no HuggingFace ZeroGPU quota limits.**

Why SF3D for Roblox UGC: it's a *direct mesh* model (not gaussian-splatting), so it emits
**one coherent mesh** instead of the fragmented splat geometry + attached backdrop planes that
TRELLIS produces. It exposes exactly the marketplace-prep controls we want:
**triangle/quad remesh**, a **vertex-count cap** (land near the Roblox tri budget), and a
**2048 texture** (the Roblox cap).

### How to run
1. **Runtime → Change runtime type → T4 GPU**, then **Save**.
2. Accept the model license once (cell 3 explains).
3. **Runtime → Run all**, upload your image when prompted, and a `.glb` downloads at the end.

First run takes ~3–5 min (install + first-time CUDA op build). After that each generation is seconds.

## 1. Check the GPU (must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Accept the license + log in to HuggingFace

SF3D's weights are **gated**. One-time steps:
1. Open <https://huggingface.co/stabilityai/stable-fast-3d> and click **Agree / Access repository**.
2. Create a **read** token at <https://huggingface.co/settings/tokens>.
3. Run the cell below and paste the token (it is not saved to the notebook).

In [ ]:
from huggingface_hub import login
import getpass
login(token=getpass.getpass('HF token (read scope): '))
print('logged in')

## 3. Install SF3D

Clones the official repo and installs deps. The custom CUDA ops (`texture_baker`,
`uv_unwrapper`) compile here with ninja — this is the slow part the first time.

In [ ]:
%cd /content
![ -d stable-fast-3d ] || git clone https://github.com/Stability-AI/stable-fast-3d.git
%cd /content/stable-fast-3d

# Colab already ships a torch matching its CUDA — do NOT reinstall torch.
# Pin setuptools: newer versions break the local C++/CUDA extension build.
!pip install -q setuptools==69.5.1 wheel ninja

# gpytoolbox 0.2.0 has no Colab wheel and fails to compile from source -> use 0.3.3.
!sed -i 's/gpytoolbox==0.2.0/gpytoolbox==0.3.3/' requirements.txt

# Build the two LOCAL CUDA ops FIRST (--no-build-isolation), forcing the T4 arch
# (sm_75 = 7.5) so they get a CUDA backend (avoids the texture_baker CUDA error).
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda'
!CUDA_HOME=/usr/local/cuda TORCH_CUDA_ARCH_LIST=7.5 pip install -q --no-build-isolation ./texture_baker/ ./uv_unwrapper/

# Strip the local ops from requirements so -r doesn't rebuild them in isolation.
# requirements.txt also (correctly) pins numpy<2 + transformers 4.42.3 etc. — SF3D's
# whole stack needs numpy<2, so DON'T force numpy>=2 anywhere. The RAPIDS/cupy
# pip warnings are harmless: SF3D never imports cudf/cuml/cupy.
!sed -i '/texture_baker/d;/uv_unwrapper/d' requirements.txt
!pip install -q -r requirements.txt

# rembg -> pymatting only OPTIONALLY needs cupy; drop the ABI-mismatched cupy and
# use a pymatting that treats the cupy GPU path as optional.
!pip uninstall -q -y cupy-cuda12x cupy
!pip install -q -U "pymatting>=1.1.12"

# Verify the pieces actually import (loud failure here if a build/dep died).
import importlib
for _mod in ('texture_baker', 'uv_unwrapper', 'rembg'):
    importlib.import_module(_mod)
print('SF3D + CUDA ops + rembg installed OK')

## 4. Upload your image

A clean subject on a plain background works best (SF3D removes the background automatically).
Run the cell, pick your file. To use a path instead, set `IMAGE_PATH` directly.

In [ ]:
from google.colab import files
import shutil, os
up = files.upload()
IMAGE_PATH = '/content/input' + os.path.splitext(next(iter(up)))[1]
shutil.move(next(iter(up)), IMAGE_PATH)
print('using', IMAGE_PATH)

## 5. Generate the mesh

Runs SF3D's Python API in **fp16 autocast** — `run.py` runs fp32 and OOMs a 16 GB T4.

Tunables:
- `REMESH` = `triangle` (clean even triangles, Roblox-friendly) / `quad` / `none`.
- `VERTEX_COUNT` = `6000` to land near the rigid-accessory budget (4,000 tris); `-1` = full detail.
- `TEXTURE_RES` = `1024` (T4-safe). 2048 is the Roblox cap but uses more VRAM — try it only if you have headroom.

In [ ]:
# SF3D Python API in fp16 autocast — this is what fits a T4 (run.py runs fp32 and OOMs).
import os, gc, torch
from PIL import Image
os.chdir("/content/stable-fast-3d")
import sf3d.utils as sf3d_utils
from sf3d.system import SF3D

REMESH = "triangle"     # triangle | quad | none  (triangle = Roblox-friendly)
VERTEX_COUNT = 6000     # cap near the rigid-accessory budget (4,000 tris); -1 = full
TEXTURE_RES = 1024      # T4-safe; bump to 2048 only if you have VRAM headroom

gc.collect(); torch.cuda.empty_cache()
model = SF3D.from_pretrained(
    "stabilityai/stable-fast-3d",
    config_name="config.yaml",
    weight_name="model.safetensors",
).to("cuda").eval()

img = Image.open(IMAGE_PATH).convert("RGBA")
img = sf3d_utils.remove_background(img, sf3d_utils.get_rembg_session())  # CPU bg removal
img = sf3d_utils.resize_foreground(img, 0.85)

with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
    mesh, _ = model.run_image(
        img, bake_resolution=TEXTURE_RES, remesh=REMESH, vertex_count=VERTEX_COUNT
    )

os.makedirs("/content/output", exist_ok=True)
GLB = "/content/output/mesh.glb"
mesh.export(GLB, include_normals=True)
print("mesh:", GLB, "| tris:", len(mesh.faces))

## 6. Quick mesh stats + download

In [ ]:
!pip install -q trimesh
import trimesh
m = trimesh.load(GLB, force='mesh')
print(f'tris: {len(m.faces):,}   verts: {len(m.vertices):,}   watertight: {m.is_watertight}')
print(f'bounds (units): {m.extents.round(3)}')
from google.colab import files
files.download(GLB)

## 7. Back in the repo

Drop the downloaded `.glb` into `runs/` and continue the pipeline:

```bash
# optional safety net (SF3D is already clean, so this should be a no-op):
roblox-ugc clean runs/shark/mesh.glb --out runs/shark/clean.glb

# import to Blender, decimate to the category tri budget, center, rescale:
roblox-ugc prep runs/shark/clean.glb --out runs/shark/prepped.fbx --decimate 4000 --center
roblox-ugc inspect runs/shark/prepped.fbx --out runs/shark/report.json
roblox-ugc validate runs/shark/report.json --target accessory --category Hat
```

Tip: pass `--target_vertex_count 6000` (cell 5) to be *born* near the rigid-accessory
4,000-tri cap, so the decimate step is light.

### Higher fidelity?
If you want richer geometry/texture and don't mind a slower run, **Hunyuan3D-2** also fits a T4
(<https://github.com/Tencent-Hunyuan/Hunyuan3D-2>) — octree mesh, also clean (no splat artifacts),
with controllable polygon count. SF3D is the fast/clean default; Hunyuan3D-2 is the quality step-up.

### ⚠️ Licensing (read before selling)
SF3D ships under the **Stability AI Community License**: free for research and for commercial use
**only if your annual revenue is ≤ US $1M** (above that needs a paid Stability Enterprise License).
The license does **not** explicitly address third-party **marketplace resale** (e.g. Roblox UGC) —
check the current `LICENSE.md` before submitting generated meshes for sale.

> Note: a **TPU** (e.g. v5e) can't run these models — they use custom CUDA kernels. Use the **T4 GPU** runtime.